<a href="https://colab.research.google.com/github/pravinjaju1005/python/blob/main/NSE_TRI_to_Gsheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-requisites

## Installing and Importing required Libraries

In [1]:
!pip install -q gspread
!pip install -q gspread-dataframe
!pip install -q nsepython

In [2]:
from nsepython import index_total_returns
from datetime import datetime
import pandas as pd

## Authentications for connecting to Google Sheets

In [3]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# NSE TRI Data Extraction

In [4]:
# Getting all Index Values in a List

def fetch_index_symbols(date: str):
    url = f"https://archives.nseindia.com/content/indices/ind_close_all_{date}.csv"
    try:
        df = pd.read_csv(url)
        # Extract unique index symbols
        symbols = df['Index Name'].dropna().unique().tolist()
        return symbols
    except Exception as e:
        print(f"❌ Failed to fetch for {date}: {e}")
        return []

# Example usage
date = "01012025"  # 1st Jan 2025
index_symbols = fetch_index_symbols(date)

In [5]:
# Getting TRI Values for all the index

start_str = "2025-01-01"
# end_str   = "2025-06-27"
end_str   = datetime.now().strftime('%Y-%m-%d')

start_date_formatted = datetime.strptime(start_str, '%Y-%m-%d').strftime("%d-%b-%Y")
end_date_formatted = datetime.strptime(end_str, '%Y-%m-%d').strftime("%d-%b-%Y")

all_raw_dfs = []
for index_sym in index_symbols:
    try:
        df_single_index = index_total_returns(index_sym, start_date_formatted, end_date_formatted)
        if df_single_index is not None and not df_single_index.empty:
            all_raw_dfs.append(df_single_index)
    except Exception:
        pass

if not all_raw_dfs:
    print("No data could be fetched for any index.")
else:
    combined_raw_df = pd.concat(all_raw_dfs, ignore_index=True)

    combined_raw_df['Date'] = pd.to_datetime(combined_raw_df['Date'], errors='coerce')
    combined_raw_df['TotalReturnsIndex'] = pd.to_numeric(combined_raw_df['TotalReturnsIndex'], errors='coerce')
    combined_raw_df.dropna(subset=['Date', 'TotalReturnsIndex'], inplace=True)

    pivoted_df = combined_raw_df.pivot_table(index='Date', columns='Index Name', values='TotalReturnsIndex')
    pivoted_df.sort_index(ascending=False, inplace=True)

In [11]:
pivoted_df.head()

Index Name,NIFTY Alpha Low-Volatility 30,NIFTY Alpha Quality Low-Volatility 30,NIFTY Alpha Quality Value Low-Volatility 30,NIFTY LargeMidcap 250,NIFTY Midcap 100,NIFTY Midcap150 Quality 50,NIFTY Quality Low-Volatility 30,NIFTY SME EMERGE,NIFTY Smallcap 100,NIFTY100 Alpha 30,...,Nifty500 Equal Weight,Nifty500 LargeMidSmall Equal-Cap Weighted,Nifty500 Low Volatility 50,Nifty500 Momentum 50,Nifty500 Multicap 50:25:25,Nifty500 Multicap India Manufacturing 50:30:20,Nifty500 Multicap Infrastructure 50:30:20,Nifty500 Multicap Momentum Quality 50,Nifty500 Quality 50,Nifty500 Shariah
Date,,,,,,,,,,,,,,,,,,,,,
2025-07-02,35355.38,30128.70,30572.80,21359.76,80712.29,33495.70,23628.90,15121.17,24427.89,23771.29,...,18933.75,23553.16,29775.52,65994.37,21403.17,19602.98,18801.95,55929.09,28003.15,9137.90
2025-07-01,35475.70,30153.98,30584.01,21401.08,80824.29,33530.96,23636.01,15176.58,24529.04,23847.67,...,18973.57,23614.14,29881.34,66476.63,21462.80,19538.79,18849.16,56294.45,28178.60,9122.50
2025-06-30,35497.80,30274.69,30719.08,21412.54,80812.33,33767.35,23735.40,15146.70,24554.01,23866.51,...,19013.99,23638.33,29816.97,66755.07,21475.89,19494.72,18817.04,56607.04,28308.69,9155.94
2025-06-27,35580.04,30333.48,30786.86,21374.37,80330.69,33518.40,23859.35,15042.15,24427.50,23786.79,...,18890.58,23541.18,29837.05,66486.55,21426.78,19476.72,18834.47,56271.89,28151.18,9125.14
2025-06-26,35449.86,30197.17,30668.54,21276.66,80117.30,33273.00,23775.18,14991.45,24205.92,23706.89,...,18772.03,23405.16,29781.21,66658.96,21306.09,19354.00,18730.26,55996.25,27974.69,9082.96


# Extracting Data to Google Sheet

In [9]:
# spreadsheet_name = 'NSE TRI Data 2025' # Replace with your sheet's name
# worksheet_name = 'Sheet1' # Replace with your desired worksheet name

# try:
#     sh = gc.open(spreadsheet_name)
#     worksheet = sh.worksheet(worksheet_name)
# except gspread.exceptions.SpreadsheetNotFound:
#     print(f"Spreadsheet '{spreadsheet_name}' not found. Please create it or check the name.")
#     # You might want to create it here if it's not found:
#     # sh = gc.create(spreadsheet_name)
#     # worksheet = sh.sheet1
# except gspread.exceptions.WorksheetNotFound:
#     print(f"Worksheet '{worksheet_name}' not found in '{spreadsheet_name}'. Creating new worksheet.")
#     worksheet = sh.add_worksheet(worksheet_name, rows="100", cols="20") # Adjust rows/cols as needed

# worksheet.clear()
# print(f"Worksheet '{worksheet_name}' in '{spreadsheet_name}' cleared.")

# from gspread_dataframe import set_with_dataframe
# set_with_dataframe(worksheet, pivoted_df, include_index=False, include_column_header=True) # Set include_index=True if you want to export the DataFrame index

# print(f"Data exported to: https://docs.google.com/spreadsheets/d/{sh.id}/edit#gid={worksheet.id}")

In [12]:
spreadsheet_name = 'NSE TRI Data 2025' # Replace with your sheet's name
worksheet_name = 'Sheet1' # Replace with your desired worksheet name

try:
    sh = gc.open(spreadsheet_name)
    worksheet = sh.worksheet(worksheet_name)
except gspread.exceptions.SpreadsheetNotFound:
    print(f"Spreadsheet '{spreadsheet_name}' not found. Creating a new spreadsheet.")
    sh = gc.create(spreadsheet_name)
    # When you create a new spreadsheet, it automatically has a 'Sheet1'.
    # We'll try to get it, or add if for some reason it's not named 'Sheet1' or we want a different name.
    try:
        worksheet = sh.worksheet(worksheet_name)
    except gspread.exceptions.WorksheetNotFound:
        print(f"Worksheet '{worksheet_name}' not found in new spreadsheet. Creating it.")
        worksheet = sh.add_worksheet(worksheet_name, rows="100", cols="20") # Adjust rows/cols as needed
except gspread.exceptions.WorksheetNotFound:
    print(f"Worksheet '{worksheet_name}' not found in '{spreadsheet_name}'. Creating new worksheet.")
    worksheet = sh.add_worksheet(worksheet_name, rows="100", cols="20") # Adjust rows/cols as needed

# --- ADD THIS LINE TO CLEAR THE SHEET ---
worksheet.clear()
print(f"Worksheet '{worksheet_name}' in '{spreadsheet_name}' cleared.")
# --- END OF ADDITION ---

from gspread_dataframe import set_with_dataframe
set_with_dataframe(worksheet, pivoted_df, include_index=True, include_column_header=True) # Set include_index=True if you want to export the DataFrame index

print(f"Data exported to: https://docs.google.com/spreadsheets/d/{sh.id}/edit#gid={worksheet.id}")

Worksheet 'Sheet1' in 'NSE TRI Data 2025' cleared.
Data exported to: https://docs.google.com/spreadsheets/d/1a_o8cKfZAUwE_F_WbkMSYeGHDdkH_2E0Rj9wpNwvwrc/edit#gid=0
